# Notebook 05 — Bayesian Belief Network & Causal Risk Inference

## Goal
Encode plausible causal hydrogeological relationships using `pgmpy`'s `DiscreteBayesianNetwork`:
- Synthesize multiple risk factors (State, Depth, Urban stress, Cluster risk, Anomaly deviance)
- Parameterize Conditional Probability Tables (CPTs) via **Maximum Likelihood Estimation (MLE)**
- Perform exact probabilistic inference with **Variable Elimination**
- Produce human-readable, transparent audit explanations for environmental health officers

## Expert Causal DAG Structure
```
  bbn_state  ─────────────────────────┐
  bbn_gwl_depth  ──────────────────────┤
  bbn_district_urban ──────────────────┼──► bbn_high_ec (Contaminated?)
  bbn_cluster_risk ────────────────────┤
  bbn_anomaly ─────────────────────────┘
```
All edges encode verified hydrogeological domain knowledge:
- **Shallow GW depth (<5m)**: higher exposure to surface runoff, agricultural leaching, and sewage
- **Urban / Industrial districts**: elevated chemical loading (industrial effluent, nitrates)
- **High-risk cluster (>=10%)**: regional geological contamination prior
- **Isolation Forest Anomaly flag**: well deviates from its local cluster peers

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
from IPython.display import Image, display

from src.bayesian_network import (
    load_anomaly_data,
    discretise_features,
    build_and_fit_bbn,
    infer_well_risk,
    explain_well,
    run_bbn_pipeline
)

PROCESSED_DIR = '../data/processed'
FIGURES_DIR   = '../figures'

## Step 1 — Load & Discretize Nodes
Convert continuous features into discrete categorical states for exact tabular Bayesian modeling.

In [ ]:
df = load_anomaly_data(PROCESSED_DIR)
df = discretise_features(df)

bbn_cols = ['bbn_high_ec', 'bbn_gwl_depth', 'bbn_state', 'bbn_district_urban', 'bbn_cluster_risk', 'bbn_anomaly']
df[bbn_cols].head(10)

## Step 2 — Empirical Marginal Contamination Rates
Examine the empirical conditional contamination probabilities across each discrete factor before fitting the DAG.

In [ ]:
display(Image(filename=os.path.join(FIGURES_DIR, 'bbn_marginal_contamination.png')))

## Step 3 — Fit Bayesian Belief Network via Maximum Likelihood Estimation (MLE)

In [ ]:
model, cpts = build_and_fit_bbn(df)
print(f"BBN Nodes fitted: {len(model.nodes())}")
print("Nodes:", model.nodes())

## Step 4 — Run Variable Elimination Inference & Explain Individual Wells
Simulate what an environmental health officer sees when triaging priority wells.

In [ ]:
# Pick a sample of high-risk and low-risk wells to explain
high_risk = df[df['bbn_cluster_risk'] == 1].head(3)
low_risk  = df[df['bbn_cluster_risk'] == 0].head(2)
sample_wells = pd.concat([high_risk, low_risk])

for idx, (_, row) in enumerate(sample_wells.iterrows(), start=1):
    explanation = explain_well(model, row)
    print(f"=== WELL SAMPLE #{idx} ===")
    print(explanation)
    print("-" * 50)

## Step 5 — Full Pipeline Execution
Compute posterior probabilities for all 36,922 wells and save `bayesian_wells.csv`.

In [ ]:
model, df_bbn, explanations = run_bbn_pipeline(PROCESSED_DIR, figures_dir=FIGURES_DIR)
print(f"Bayesian dataset generated with {len(df_bbn):,} evaluated wells.")
df_bbn[['station', 'state', 'district', 'bbn_prob_contaminated', 'is_contaminated']].head()